In [28]:
!pip install fastapi uvicorn nest-asyncio transformers torch pydantic requests

In [29]:
%%writefile requirements.txt
fastapi
uvicorn
nest-asyncio
transformers
torch
pydantic
requests
Pillow
python-multipart

Overwriting requirements.txt


In [30]:
!pip install -r requirements.txt

In [31]:
%%writefile main.py
from fastapi import FastAPI, HTTPException, File, UploadFile
from transformers import pipeline
from PIL import Image
import io

app = FastAPI(
    title="Vision Transformer API",
    description="API phân loại hình ảnh sử dụng mô hình ViT của Google"
)

# Khởi tạo pipeline phân loại hình ảnh
try:
    classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
except Exception as e:
    classifier = None
    print(f"Lỗi khi tải mô hình: {e}")

@app.get("/")
def read_root():
    return {
        "name": "Hệ thống Phân loại Hình ảnh API",
        "description": "API này nhận vào một tệp hình ảnh và trả về kết quả dự đoán (ví dụ: con mèo, ô tô,...) cùng với độ tin cậy."
    }

@app.get("/health")
def health_check():
    if classifier is None:
        raise HTTPException(status_code=503, detail="Mô hình chưa sẵn sàng hoạt động")
    return {"status": "Hoạt động bình thường"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    # Kiểm tra định dạng file
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Vui lòng tải lên một tệp hình ảnh hợp lệ (JPEG, PNG,...).")

    try:
        # Đọc dữ liệu byte từ file tải lên
        image_data = await file.read()

        # Chuyển đổi thành đối tượng hình ảnh PIL
        image = Image.open(io.BytesIO(image_data))

        # Gọi mô hình Hugging Face để suy luận
        result = classifier(image)
        return {"data": result}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi xử lý hình ảnh: {str(e)}")

Overwriting main.py


In [32]:
import nest_asyncio
import uvicorn
import threading
import time

nest_asyncio.apply()

def run_server():

    uvicorn.run("main:app", host="0.0.0.0", port=1234, reload=False)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()


time.sleep(3)
print("Server FastAPI đang chạy ngầm ..")

INFO:     Started server process [12219]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 1234): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server FastAPI đang chạy ngầm ..


In [33]:
# Chạy lệnh này, Colab sẽ hiển thị một đường link Public.
#ssh -p 443 -R0:localhost:1234 qr@a.pinggy.io

In [38]:
%%writefile test_api.py
import requests

BASE_URL = "http://ubrnx-136-113-169-159.run.pinggy-free.link"

print("--- 1. Kiểm tra giới thiệu API (GET /) ---")
try:
    res = requests.get(f"{BASE_URL}/")
    print(res.json())
except Exception as e:
    print("Lỗi kết nối:", e)

print("\n--- 2. Kiểm tra Health Check (GET /health) ---")
try:
    res = requests.get(f"{BASE_URL}/health")
    print(res.json())
except Exception as e:
    print("Lỗi kết nối:", e)

print("\n--- 3. Kiểm tra Suy luận (POST /predict) với ảnh từ máy ---")



image_list = ["Image_1.jpg", "Image_2.jpg"]
for image_filename in image_list:
    print(f"\n>> Đang xử lý file: {image_filename}")
    try:

        with open(image_filename, "rb") as image_file:
            # Cấu trúc dữ liệu gửi lên API
            files = {"file": (image_filename, image_file, "image/jpeg")}

            # Gửi request
            res = requests.post(f"{BASE_URL}/predict", files=files)

            print(f"Kết quả dự đoán cho {image_filename}:")
            print(res.json())

    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{image_filename}'. Bạn hãy kiểm tra lại tên file hoặc chắc chắn đã upload lên Colab chưa nhé.")
    except Exception as e:
        print("Lỗi:", e)

Overwriting test_api.py


In [39]:
!python test_api.py

--- 1. Kiểm tra giới thiệu API (GET /) ---
INFO:     136.113.169.159:0 - "GET / HTTP/1.1" 200 OK
{'name': 'Hệ thống Phân loại Hình ảnh API', 'description': 'API này nhận vào một tệp hình ảnh và trả về kết quả dự đoán (ví dụ: con mèo, ô tô,...) cùng với độ tin cậy.'}

--- 2. Kiểm tra Health Check (GET /health) ---
INFO:     136.113.169.159:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'Hoạt động bình thường'}

--- 3. Kiểm tra Suy luận (POST /predict) với ảnh từ máy ---

>> Đang xử lý file: Image_1.jpg
INFO:     136.113.169.159:0 - "POST /predict HTTP/1.1" 200 OK
Kết quả dự đoán cho Image_1.jpg:
{'data': [{'label': 'comic book', 'score': 0.9917703866958618}, {'label': 'book jacket, dust cover, dust jacket, dust wrapper', 'score': 0.004874749109148979}, {'label': 'breastplate, aegis, egis', 'score': 0.0003304460260551423}, {'label': 'mask', 'score': 0.0003245098632760346}, {'label': 'ski mask', 'score': 0.0001313259417656809}]}

>> Đang xử lý file: Image_2.jpg
INFO:     136.113.169.159:0 -